In [1]:
import sys
import subprocess

# 1. Force-Check Versions (Safety Step)
# If the wrong versions are still there, this will fix them automatically.
try:
    import numpy
    import cv2
    if numpy.__version__.startswith("2") or cv2.__version__.startswith("4.12"):
        print("⚠️ Found incompatible versions. Fixing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2.0", "opencv-python-headless<4.10", "ultralytics"])
        print("✅ Fixed! Please RESTART THE SESSION one last time.")
        exit() # Stops execution so you can restart
except ImportError:
    pass

# 2. Main Training Code
import yaml
import shutil
import os
from ultralytics import YOLO

print("✅ Environment is clean. Starting training...")

# Define dataset path
dataset_path = '/kaggle/input/top-view-dataset'

# Create config
yaml_content = {
    'train': f'{dataset_path}/train/images',
    'val':   f'{dataset_path}/valid/images',
    'test':  f'{dataset_path}/test/images',
    'nc': 1,
    'names': ['person']
}

with open('my_config.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

# Train
model = YOLO('yolov8n.pt')

model.train(
    data='my_config.yaml', 
    epochs=50, 
    imgsz=640, 
    batch=-1, 
    cache=True, 
    name='my_run', 
    exist_ok=True
)

# Save
if os.path.exists('runs/detect/my_run/weights/best.pt'):
    shutil.copy('runs/detect/my_run/weights/best.pt', 'final_model.pt')
    print("\n🎉 SUCCESS! You can now download 'final_model.pt' from the Output tab.")

✅ Environment is clean. Starting training...
Ultralytics 8.3.237 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=my_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=my_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

In [3]:
import glob
import time
import numpy as np
import yaml
from ultralytics import YOLO

# 1. MAP YOUR SPECIFIC PATH (Safety Step)
dataset_path = '/kaggle/input/top-view-dataset'

# Ensure config points to the right Kaggle folders
yaml_content = {
    'train': f'{dataset_path}/train/images',
    'val':   f'{dataset_path}/valid/images',
    'test':  f'{dataset_path}/test/images',
    'nc': 1, 'names': ['person']
}
with open('my_config.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

# 2. LOAD MODEL
model = YOLO('final_model.pt')

# 3. VALIDATION (Calculates Accuracy on Test Data)
print("\n--- 1. VALIDATION RESULTS ---")
metrics = model.val(data='my_config.yaml', split='test')
print(f"✅ mAP50 Accuracy: {metrics.box.map50:.3f}")

# 4. VISUAL TEST (Predict on images from your path)
print("\n--- 2. VISUAL PREDICTIONS ---")
# Uses your path to find test images
test_images = glob.glob(f'{dataset_path}/test/images/*.jpg')[:3] 

if test_images:
    model.predict(test_images, save=True, conf=0.25)
    print("✅ Prediction images saved in 'runs/detect/predict'")
else:
    print("⚠️ No images found in test folder.")

# 5. FPS & LATENCY (Speed Test)
print("\n--- 3. SPEED BENCHMARK ---")
dummy_img = np.zeros((640, 640, 3), dtype=np.uint8)

# Warmup GPU
for _ in range(10): model(dummy_img, verbose=False)

# Measure 100 frames
start = time.time()
for _ in range(100): model(dummy_img, verbose=False)
end = time.time()

print(f"🚀 FPS: {100 / (end - start):.2f}")
print(f"⏱️ Latency: {((end - start) / 100) * 1000:.2f} ms")


--- 1. VALIDATION RESULTS ---
Ultralytics 8.3.237 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 276.5±119.1 MB/s, size: 118.9 KB)
val: Scanning /kaggle/input/top-view-dataset/test/labels... 52 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 52/52 1.0Kit/s 0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/top-view-dataset/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.4it/s 2.8s1.0s
                   all         52        677      0.953      0.894      0.952      0.649
Speed: 5.0ms preprocess, 9.4ms inference, 0.0ms loss, 8.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
✅ mAP50 Accuracy: 0.952

--- 2. VISUAL PREDICTIONS ---

0: 640x640 3 persons, 5.8ms
1: 640x640 38 persons, 5.8ms
2: 640x640 4 persons, 